In [1]:
seed = 2
import numpy as np
np.random.seed(seed)

import warnings
warnings.filterwarnings("ignore")

import tensorflow.compat.v2 as tf
# V2 Behavior is necessary to use TF2 APIs before TF2 is default TF version internally.
tf.enable_v2_behavior()
tf.random.set_seed(seed)

import sys
sys.path.append("/home/george/Desktop/workspace/dnn_processor/")  # just to enable `dataset`
sys.path.append("/home/george/Desktop/workspace/dnn_processor/dataset/")  # just to enable `dataset`
sys.path.append("")


import dataset
import numpy as np
import keras
from datetime import datetime
from keras import layers
import qkeras
from qkeras.autoqkeras.utils import print_qmodel_summary
from qkeras.autoqkeras import AutoQKeras
import time

2025-04-13 19:32:36.260268: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-13 19:32:36.261885: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-13 19:32:36.300476: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-13 19:32:36.301385: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-13 19:32:36.779834: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

In [2]:
DEFAULT_BATCH_SIZE = 256
EPOCHS = 100
DEFAULT_INPUT_SIZE = 128

train_images_rgx = "../../dataset/train_set/*/*.jpeg"
test_images_rgx = "../../dataset/test_set/*/*.jpeg"

train_dataset, test_dataset, num_classes = dataset.create_datasets(
    train_images_rgx, test_images_rgx, DEFAULT_BATCH_SIZE, DEFAULT_INPUT_SIZE
)

2025-04-13 19:32:37.884840: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-04-13 19:32:37.908810: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1956] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


# load the base line model

In [3]:
EXP_ID = "2x_wider_baseline_model"

baseline_model = keras.models.load_model(
    f"../handmade/saved_models/{EXP_ID}/best_model_t/model_clean.h5",
    compile=False,
)

## ADD A LINEAR ACTIVATION LAYER BEFORE FLATTEN
x = x_in = keras.Input(shape=(128,128,3))

x = keras.layers.Activation("linear", name="qtz_inputs")(x)

for l in baseline_model.layers[1:]:
    if isinstance(l, keras.layers.Flatten):
        x = keras.layers.Activation("linear", name="qtz_flatten_input")(x)
    
    x = l(x)

model = keras.Model(x_in, x)

model.compile(
    keras.optimizers.Adam(), loss=tf.keras.losses.CategoricalCrossentropy(True), metrics=["accuracy"]
)

model.save("tempmodel.h5")


model = keras.models.load_model("tempmodel.h5", compile=False)
model.compile(
    keras.optimizers.Adam(), loss=tf.keras.losses.CategoricalCrossentropy(True), metrics=["accuracy"]
)


score = model.evaluate(test_dataset)
print("Accuracy: {}".format(score[1]))

model.summary()
baseline_model.summary()

1/1 [==============================] - 0s 197ms/step - loss: 0.1127 - accuracy: 1.0000
Accuracy: 1.0
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128, 128, 3)]     0         
                                                                 
 qtz_inputs (Activation)     (None, 128, 128, 3)       0         
                                                                 
 conv2d (Conv2D)             (None, 32, 32, 8)         392       
                                                                 
 activation (Activation)     (None, 32, 32, 8)         0         
                                                                 


2025-04-13 19:32:38.234254: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [9]
	 [[{{node Placeholder/_0}}]]
2025-04-13 19:32:38.234486: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [9]
	 [[{{node Placeholder/_0}}]]


 conv2d_1 (Conv2D)           (None, 15, 15, 32)        2336      
                                                                 
 activation_1 (Activation)   (None, 15, 15, 32)        0         
                                                                 
 conv2d_2 (Conv2D)           (None, 15, 15, 10)        330       
                                                                 
 qtz_flatten_input (Activati  (None, 15, 15, 10)       0         
 on)                                                             
                                                                 
 flatten (Flatten)           (None, 2250)              0         
                                                                 
 dense (Dense)               (None, 5)                 11255     
                                                                 
Total params: 14,313
Trainable params: 14,313
Non-trainable params: 0
_________________________________________________________________
Model:


# Quantizing a Model With AutoQKeras

To quantize this model with AutoQKeras, we need to define the quantization for kernels, biases and activations; forgiving factors and quantization strategy.

Below we define which quantizers are allowed for kernel, bias, activations and linear. Linear is a proxy that we use to capture Activation("linear") to apply quantization without applying a non-linear operation. In some networks, we found that this trick may be necessary to better represent the quantization space.


In [4]:
# These are the quantizers we'll test in the bayesian optimization
quantization_config = {
    "kernel": {
        "quantized_bits(2,0,1,alpha=1.0)": 2,
        "quantized_bits(4,0,1,alpha=1.0)": 4,
        "quantized_bits(6,0,1,alpha=1.0)": 6,
        "quantized_bits(8,0,1,alpha=1.0)": 8,
    },
    "bias": {
        "quantized_bits(2,0,1,alpha=1.0)": 2,
        "quantized_bits(4,0,1,alpha=1.0)": 4,
        "quantized_bits(6,0,1,alpha=1.0)": 6,
        "quantized_bits(8,0,1,alpha=1.0)": 8,
    },
    "activation": {
        "quantized_relu(3,1)": 3,
        "quantized_relu(4,2)": 4,
        "quantized_relu(8,2)": 8,
        "quantized_relu(8,4)": 8,        
    },
    "linear": {
        "quantized_bits(2,0,1,alpha=1.0)": 2,
        "quantized_bits(4,0,1,alpha=1.0)": 4,
        "quantized_bits(6,0,1,alpha=1.0)": 6,
        "quantized_bits(8,0,1,alpha=1.0)": 8,
    },
}

# These are the layer types we will quantize
limit = {
    "Dense": [8, 8, 8],
    "Conv2D": [8, 8, 8],
    "Activation": [8],
}

# Use this if you want to minimize the model bit size
goal_bits = {
    "type": "bits",
    "params": {
        "delta_p": 5.0,  # We tolerate up to a +5% accuracy change
        "delta_n": 5.0,  # We tolerate down to a -5% accuracy change
        "rate": 2.0,  # We want a x2 times smaller model
        "stress": 1.0,  # Force the reference model size to be smaller by setting stress<1
        "input_bits": 8,
        "output_bits": 8,
        "ref_bits": 8,
        "config": {"default": ["parameters", "activations"]},
    },
}

# Use this if you want to minimize the model energy consumption
goal_energy = {
    "type": "energy",
    "params": {
        "delta_p": 5.0,
        "delta_n": 5.0,
        "rate": 1.1,
        "stress": 1.0,
        "process": "horowitz",
        "parameters_on_memory": ["sram", "sram"],
        "activations_on_memory": ["sram", "sram"],
        "rd_wr_on_io": [False, False],
        "min_sram_size": [0, 0],
        "source_quantizers": ["fp32"],
        "reference_internal": "int8",
        "reference_accumulator": "int32",
    },
}

run_config = {    
    "goal": goal_bits,
    "quantization_config": quantization_config,
    "learning_rate_optimizer": False,
    "transfer_weights": True,  # Randomely initialize weights
    "mode": "bayesian",  # This can be bayesian,random,hyperband
    "seed": 42,
    "limit": limit,
    "tune_filters": "none",
    "tune_filters_exceptions": "^dense|^conv2d_2",
    "distribution_strategy": None,
    "max_trials": 200,  # Let's just do 5 trials for this demonstrator, ideally you should do as many as possible
}


In [5]:
import qkeras
from qkeras.autoqkeras.utils import print_qmodel_summary
from qkeras.autoqkeras import AutoQKeras
import time

from qkeras.utils import _add_supported_quantized_objects

co = {}
_add_supported_quantized_objects(co)

In [6]:
autoqk = AutoQKeras(
    model, output_dir="autoqqatwqkeras", metrics=["acc"], custom_objects=co, **run_config
)
autoqk.fit(train_dataset, validation_data=test_dataset, epochs=30)


Trial 200 Complete [00h 00m 03s]
val_score: 0.7579799294471741

Best val_score So Far: 0.9876401424407959
Total elapsed time: 00h 22m 37s


In [7]:
aqmodel = autoqk.get_best_model()
aqmodel.save_weights("autoqqatwqkeras_v2/bestmodel/model_weights.h5")
print_qmodel_summary(aqmodel)

learning_rate: 0.0010000000474974513
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128, 128, 3)]     0         
                                                                 
 qtz_inputs (QActivation)    (None, 128, 128, 3)       0         
                                                                 
 conv2d (QConv2D)            (None, 32, 32, 8)         392       
                                                                 
 activation (QActivation)    (None, 32, 32, 8)         0         
                                                                 
 conv2d_1 (QConv2D)          (None, 15, 15, 32)        2336      
                                                                 
 activation_1 (QActivation)  (None, 15, 15, 32)        0         
                                                                 
 conv2d_2 (QConv2D)     

In [8]:
layers = [l for l in aqmodel.layers]
x = layers[0].output
for i in range(1, len(layers)):
    x = layers[i](x)

new_model = keras.Model(inputs=[layers[0].input], outputs=[x])
LOSS = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
OPTIMIZER = tf.keras.optimizers.Adam(learning_rate=0.0005)

new_model.compile(loss=LOSS, optimizer=OPTIMIZER, metrics=["accuracy"])
new_model.summary()
new_model.load_weights("autoqqatwqkeras_v2/bestmodel/model_weights.h5")
new_model.save("autoqqatwqkeras_v2/bestmodel/model.h5")
print_qmodel_summary(new_model)
from qkeras.estimate import print_qstats
print_qstats(new_model)

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128, 128, 3)]     0         
                                                                 
 qtz_inputs (QActivation)    (None, 128, 128, 3)       0         
                                                                 
 conv2d (QConv2D)            (None, 32, 32, 8)         392       
                                                                 
 activation (QActivation)    (None, 32, 32, 8)         0         
                                                                 
 conv2d_1 (QConv2D)          (None, 15, 15, 32)        2336      
                                                                 
 activation_1 (QActivation)  (None, 15, 15, 32)        0         
                                                                 
 conv2d_2 (QConv2D)          (None, 15, 15, 10)        330   

Instructions for updating:
Use ref() instead.



Number of operations in model:
    conv2d                        : 393216 (smult_8_4)
    conv2d_1                      : 518400 (smult_4_8)
    conv2d_2                      : 72000 (smult_8_4)
    dense                         : 11250 (smult_4_2)

Number of operation types in model:
    smult_4_2                     : 11250
    smult_4_8                     : 518400
    smult_8_4                     : 465216

Weight profiling:
    conv2d_weights                 : 384   (8-bit unit)
    conv2d_bias                    : 8     (4-bit unit)
    conv2d_1_weights               : 2304  (4-bit unit)
    conv2d_1_bias                  : 32    (6-bit unit)
    conv2d_2_weights               : 320   (8-bit unit)
    conv2d_2_bias                  : 10    (2-bit unit)
    dense_weights                  : 11250 (4-bit unit)
    dense_bias                     : 5     (2-bit unit)

Weight sparsity:
... quantizing model
    conv2d                         : 0.0204
    conv2d_1                       

In [9]:
new_model = qkeras.utils.load_qmodel("autoqqatwqkeras_v2/bestmodel/model.h5")

In [10]:
model.evaluate(test_dataset)
new_model.evaluate(test_dataset)

1/1 [==============================] - 0s 239ms/step - loss: 0.6543 - accuracy: 0.7778


[0.6542748212814331, 0.7777777910232544]

In [11]:
new_model.fit(train_dataset, batch_size=256, validation_data=test_dataset, epochs=10)
new_model.evaluate(test_dataset)

Epoch 1/10
2/2 [==============================] - 1s 118ms/step - loss: 1.5538 - accuracy: 0.6680 - val_loss: 0.9258 - val_accuracy: 0.7778
Epoch 2/10
2/2 [==============================] - 0s 108ms/step - loss: 0.6399 - accuracy: 0.7617 - val_loss: 0.3333 - val_accuracy: 0.8889
Epoch 3/10
2/2 [==============================] - 0s 99ms/step - loss: 0.4872 - accuracy: 0.8242 - val_loss: 0.3299 - val_accuracy: 0.7778
Epoch 4/10
2/2 [==============================] - 0s 99ms/step - loss: 0.5699 - accuracy: 0.8047 - val_loss: 0.1837 - val_accuracy: 1.0000
Epoch 5/10
2/2 [==============================] - 0s 98ms/step - loss: 0.5135 - accuracy: 0.8242 - val_loss: 0.1684 - val_accuracy: 1.0000
Epoch 6/10
2/2 [==============================] - 0s 96ms/step - loss: 0.4713 - accuracy: 0.8340 - val_loss: 0.1809 - val_accuracy: 0.8889
Epoch 7/10
2/2 [==============================] - 0s 97ms/step - loss: 0.3668 - accuracy: 0.8594 - val_loss: 0.1021 - val_accuracy: 1.0000
Epoch 8/10
2/2 [=========

[0.044605180621147156, 1.0]

In [12]:
new_model.save("autoqqatwqkeras_v2/bestmodel/new_model.h5")

# check metrics

In [13]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

x_test = []
y_test = []
for b in test_dataset.as_numpy_iterator():
    x_test.append(b[0])
    y_test.append(b[1])

x_test = np.concatenate(x_test, 0)
y_test = np.concatenate(y_test, 0)

y_test = np.argmax(y_test, axis=1)
y_qpred = new_model.predict(x_test)
y_qpred = np.argmax(y_qpred, axis=1)

print("accuracy_score", accuracy_score(y_test, y_qpred))
print("precision_score", precision_score(y_test, y_qpred, average="macro"))
print("recall_score", recall_score(y_test, y_qpred, average="macro"))
print("f1_score", f1_score(y_test, y_qpred, average="macro"))


1/1 [==============================] - 0s 210ms/step
accuracy_score 1.0
precision_score 1.0
recall_score 1.0
f1_score 1.0
